# Water Quality Indices from EO-1 Hyperion L1T

**Sensor:** EO-1 Hyperion — 242 bands, 356–2577 nm, 10 nm sampling, 30 m spatial resolution  
**Scene:** Lake Garda, Italy — 7 October 2002  
**Processing level:** L1T (terrain-corrected radiance, DN)

### Indices computed
| Index | Target parameter |
|---|---|
| NDWI | Water body mask |
| NDCI | Chlorophyll-*a* proxy |
| Three-band model (Dall'Olmo / Gitelson) | Chlorophyll-*a* proxy |
| FAI | Floating algae / cyanobacteria surface scum |
| Phycocyanin index | Cyanobacteria |
| Fluorescence Line Height (FLH) | Phytoplankton biomass |
| CDOM proxy | Dissolved organic matter |

### ⚠️ Important limitations

**Atmospheric correction:** This notebook works with at-sensor radiance (DN ÷ scaling factor), not surface reflectance.
For quantitatively accurate results, atmospheric correction should be applied first to convert to surface reflectance / Rrs.
Recommended tools:
- **ENVI FLAASH or ATCOR** — physics-based, best suited for Hyperion
- **QUAC** (ENVI) — scene-based, faster but less accurate
- **Py6S** — open-source Python interface to the 6S radiative transfer model

Atmospheric effects partially cancel in band ratios, so the spatial patterns shown here are
meaningful for learning and exploratory analysis, but absolute values are not reliable without atmospheric correction.

**Cloud masking:** No cloud mask is applied. Clouds and cloud shadows should be removed
before computing indices for accurate results. For Hyperion, options include:
- Manual radiance threshold (clouds are bright and spectrally flat)
- ENVI's cloud masking utilities

Visually inspect the RGB composite (Cell 4) before proceeding.

### Requirements
```
pip install rasterio numpy matplotlib
```


## 1  Imports and data folder

In [ ]:
import os
import glob
import numpy as np
import rasterio
import matplotlib.pyplot as plt

# Path to the folder containing the unzipped Hyperion GeoTIFF band files
data_dir = "./hyperion_data"   # ← change to your folder path


## 2  Hyperion band reference

Hyperion delivers 242 GeoTIFF files, one per band.
Only **bands 8–57** (~427–925 nm, VNIR) are radiometrically calibrated and suitable for water quality work.

| Bands | Spectral range | Scale factor | Status |
|---|---|---|---|
| B001–B007 | 356–417 nm | ÷ 40 | ❌ uncalibrated |
| **B008–B057** | **427–925 nm** | **÷ 40** | ✅ use these |
| B058–B070 | 935–1054 nm | ÷ 40 | ❌ noisy VNIR/SWIR overlap |
| B071–B242 | 852–2577 nm (SWIR) | ÷ 80 | ⚠️ loaded separately for FAI only |

Centre wavelength (nm) = **356.9 + (band_number − 1) × 10.0**


In [ ]:
GOOD_VNIR_BANDS = list(range(8, 58))   # 50 calibrated VNIR bands
VNIR_SCALE      = 40.0                 # DN ÷ 40 → radiance (W m⁻² sr⁻¹ μm⁻¹)

def band_wavelength(band_number):
    return 356.9 + (band_number - 1) * 10.0

wavelengths = np.array([band_wavelength(b) for b in GOOD_VNIR_BANDS])
print("Wavelength range:", wavelengths[0], "–", wavelengths[-1], "nm")


## 3  Load bands → 3-D radiance cube

All 50 VNIR GeoTIFFs are loaded and stacked into a single array of shape **(bands, rows, cols)**.  
DN = 0 is the Hyperion no-data value and is replaced with NaN.


In [ ]:
bands_list = []

for b in GOOD_VNIR_BANDS:
    pattern = os.path.join(data_dir, f"*_B{b:03d}_L1T.TIF")
    files = glob.glob(pattern)

    with rasterio.open(files[0]) as src:
        data = src.read(1).astype(np.float32)
        profile = src.profile   # spatial metadata (CRS, transform) kept for potential export

    data[data == 0] = np.nan    # no-data → NaN
    data = data / VNIR_SCALE    # DN → radiance
    bands_list.append(data)

cube = np.stack(bands_list, axis=0)   # shape: (50, rows, cols)
print("Cube shape (bands, rows, cols):", cube.shape)


## 4  Band selection helper and RGB preview

`get_band(target_nm)` returns the 2-D radiance array whose centre wavelength is
closest to `target_nm`. This avoids hardcoding band indices throughout the notebook.


In [ ]:
def get_band(target_nm):
    """Return the 2-D radiance array from cube closest to target_nm (nm)."""
    idx = np.argmin(np.abs(wavelengths - target_nm))
    return cube[idx]


### RGB composite

Inspect the scene before computing any index.  
A 2–98 percentile stretch is applied per channel so the image is not washed out by outliers.


In [ ]:
def stretch(band):
    """Linearly stretch a band to 0–1 using the 2nd–98th percentile range."""
    lo, hi = np.nanpercentile(band, [2, 98])
    return np.clip((band - lo) / (hi - lo), 0, 1)

rgb = np.stack([stretch(get_band(660)),
                stretch(get_band(560)),
                stretch(get_band(470))], axis=2)
rgb = np.nan_to_num(rgb)   # NaN → 0 (imshow does not accept NaN in RGB arrays)

plt.figure(figsize=(5, 8))
plt.imshow(rgb)
plt.title("RGB composite — Lake Garda, 7 Oct 2002")
plt.axis("off")
plt.tight_layout()
plt.show()


\
## 5  Water mask (NDWI)

$$NDWI = \frac{R(560) - R(860)}{R(560) + R(860)}$$

**Typical NDWI values:**
| NDWI | Interpretation |
|---|---|
| > 0.3 | Open water (threshold used here) |
| 0.0 – 0.3 | Moist soil / shallow water |
| < 0.0 | Vegetation / bare land |


In [ ]:
green = get_band(560)
nir   = get_band(860)

NDWI = (green - nir) / (green + nir)
water_mask = NDWI > 0.3

# Apply mask to the full cube: land pixels → NaN
cube_water = np.where(water_mask[np.newaxis, :, :], cube, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(NDWI, cmap="RdBu", vmin=-0.5, vmax=0.5)
axes[0].set_title("NDWI")
axes[0].axis("off")
axes[1].imshow(water_mask, cmap="Blues")
axes[1].set_title("Water mask (NDWI > 0.3)")
axes[1].axis("off")
plt.tight_layout()
plt.show()


## 6  Median radiance spectrum over water pixels

A sanity check of the loaded data. Expected shape for a productive lake:
- High in blue (~430–500 nm) — partly from Rayleigh atmospheric scattering
- Decreasing through green and red
- Small bump near 700–710 nm — chlorophyll-*a* fluorescence peak
- Near zero beyond ~750 nm — strong NIR absorption by water


In [ ]:
median_spectrum = np.nanmedian(cube_water, axis=(1, 2))

plt.figure(figsize=(10, 4))
plt.plot(wavelengths, median_spectrum, color="steelblue")
plt.xlabel("Wavelength (nm)")
plt.ylabel("Radiance (W m⁻² sr⁻¹ μm⁻¹)")
plt.title("Median radiance spectrum — water pixels only")
plt.grid(True)
plt.tight_layout()
plt.show()


\
## 7  NDCI — Normalized Difference Chlorophyll Index

$$NDCI = \frac{R(709) - R(665)}{R(709) + R(665)}$$

**Interpretation (radiance-based, approximate):**
| NDCI | Indication |
|---|---|
| < 0.0 | Low chlorophyll / clear water |
| 0.0 – 0.1 | Moderate productivity |
| > 0.1 | Elevated chlorophyll — potential bloom risk |

> Quantitative thresholds require atmospheric correction and empirical calibration.


In [ ]:
red      = get_band(665)
red_edge = get_band(709)

NDCI = (red_edge - red) / (red_edge + red)
NDCI_water = np.where(water_mask, NDCI, np.nan)

vmin, vmax = np.nanpercentile(NDCI_water, [2, 98])

plt.figure(figsize=(5, 8))
im = plt.imshow(NDCI_water, cmap="RdYlGn", vmin=vmin, vmax=vmax)
plt.colorbar(im, label="NDCI")
plt.title("NDCI — Chlorophyll-a proxy")
plt.axis("off")
plt.tight_layout()
plt.show()


\
## 8  Three-band Chl-a model (Dall'Olmo / Gitelson)

$$Chl_a \propto \left(\frac{1}{R(670)} - \frac{1}{R(710)}\right) \cdot R(740)$$

Output is a dimensionless proxy. An empirical regression against field measurements
is needed to convert to mg m⁻³. Spatial patterns (e.g. gradients near tributaries)
are more interpretable than absolute values without atmospheric correction.


In [ ]:
r670 = get_band(670)
r710 = get_band(710)
r740 = get_band(740)

chl = ((1 / r670) - (1 / r710)) * r740
chl_water = np.where(water_mask, chl, np.nan)

vmin, vmax = np.nanpercentile(chl_water, [2, 98])

plt.figure(figsize=(5, 8))
im = plt.imshow(chl_water, cmap="YlGn", vmin=vmin, vmax=vmax)
plt.colorbar(im, label="Three-band index")
plt.title("Three-band Chl-a model")
plt.axis("off")
plt.tight_layout()
plt.show()


\
## 9  FAI — Floating Algae Index

$$FAI = R(860) - \left[ R(665) + \left(R(1245) - R(665)\right) \cdot \frac{860 - 665}{1245 - 665} \right]$$

FAI uses a SWIR band (~1245 nm, Hyperion B089), loaded separately with its own scaling factor (÷ 80).

**Interpretation:**
| FAI | Indication |
|---|---|
| < 0 | Open water, no surface algae |
| ≈ 0 | Sparse / marginal floating material |
| > 0 | Floating algae or cyanobacterial surface scum |


In [ ]:
swir_files = glob.glob(os.path.join(data_dir, "*_B089_L1T.TIF"))

with rasterio.open(swir_files[0]) as src:
    swir_dn = src.read(1).astype(np.float32)

swir_dn[swir_dn == 0] = np.nan
swir = swir_dn / 80.0   # SWIR scaling factor

fai = nir - (red + (swir - red) * ((860 - 665) / (1245 - 665)))
fai_water = np.where(water_mask, fai, np.nan)

vmin, vmax = np.nanpercentile(fai_water, [2, 98])

plt.figure(figsize=(5, 8))
im = plt.imshow(fai_water, cmap="BuPu", vmin=vmin, vmax=vmax)
plt.colorbar(im, label="FAI")
plt.title("FAI — Floating Algae Index")
plt.axis("off")
plt.tight_layout()
plt.show()


\
## 10  Phycocyanin index

Based on Simis et al. (2005). Phycocyanin is the pigment specific to cyanobacteria
(blue-green algae), making this index useful for detecting potentially toxic blooms.

$$PC \propto \frac{R(709)}{R(620)} \cdot R(665)$$

This index requires the narrow (~10 nm) bands of Hyperion and is not computable
from standard multispectral sensors such as Sentinel-2 or Landsat.

**Interpretation:**
Higher values indicate greater cyanobacterial presence.

> ⚠️ For health-risk assessment, PC proxy values must be calibrated against field measurements of phycocyanin concentration (μg L⁻¹).
> WHO guideline levels: > 15 μg L⁻¹ = moderate risk; > 100 μg L⁻¹ = high risk.


In [ ]:
r620 = get_band(620)
r709 = get_band(709)
r665 = get_band(665)

pc = (r709 / r620) * r665
pc_water = np.where(water_mask, pc, np.nan)

vmin, vmax = np.nanpercentile(pc_water, [2, 98])

plt.figure(figsize=(5, 8))
im = plt.imshow(pc_water, cmap="Oranges", vmin=vmin, vmax=vmax)
plt.colorbar(im, label="Phycocyanin proxy")
plt.title("Phycocyanin index (cyanobacteria)")
plt.axis("off")
plt.tight_layout()
plt.show()


\
## 11  Fluorescence Line Height (FLH)

FLH measures the chlorophyll-*a* fluorescence emission peak at ~678 nm above a
linear baseline drawn between 667 nm and 746 nm.

$$FLH = R(678) - R(667) - \left(R(746) - R(667)\right) \cdot \frac{678 - 667}{746 - 667}$$

This index requires narrow bands to resolve the fluorescence peak and is not
reliably computable from multispectral sensors.

**Interpretation:**
| FLH | Indication |
|---|---|
| ≈ 0 or negative | Low phytoplankton biomass |
| Small positive | Moderate productivity |
| Large positive | High biomass — bloom conditions |


In [ ]:
r667 = get_band(667)
r678 = get_band(678)
r746 = get_band(746)

flh = r678 - r667 - (r746 - r667) * ((678 - 667) / (746 - 667))
flh_water = np.where(water_mask, flh, np.nan)

vmin, vmax = np.nanpercentile(flh_water, [2, 98])

plt.figure(figsize=(5, 8))
im = plt.imshow(flh_water, cmap="YlOrRd", vmin=vmin, vmax=vmax)
plt.colorbar(im, label="FLH")
plt.title("Fluorescence Line Height (FLH)")
plt.axis("off")
plt.tight_layout()
plt.show()


\
## 12  CDOM proxy

Coloured Dissolved Organic Matter (CDOM) absorbs strongly at short blue wavelengths.
A simple proxy is the ratio of blue (~412 nm) to green (~555 nm) radiance.

$$CDOM \approx \frac{R(412)}{R(555)}$$

> ⚠️ The shortest calibrated Hyperion VNIR band is ~427 nm (B008).
> `get_band(412)` will return that band (~427 nm). Treat this map as indicative only.

**Interpretation:**
Lower ratio → stronger blue absorption → more CDOM.  
Higher ratio → clearer, less organic water.


In [ ]:
r412 = get_band(412)   # returns ~427 nm — closest calibrated band
r555 = get_band(555)

cdom = r412 / r555
cdom_water = np.where(water_mask, cdom, np.nan)

vmin, vmax = np.nanpercentile(cdom_water, [2, 98])

plt.figure(figsize=(5, 8))
im = plt.imshow(cdom_water, cmap="YlOrBr", vmin=vmin, vmax=vmax)
plt.colorbar(im, label="CDOM proxy (R427/R555)")
plt.title("CDOM proxy")
plt.axis("off")
plt.tight_layout()
plt.show()


## 13  Summary panel — all indices

In [ ]:
indices = [
    (NDWI,       "NDWI",             "RdBu",    -0.5, 0.5),
    (NDCI_water, "NDCI",             "RdYlGn",  *[np.nanpercentile(NDCI_water, p) for p in [2, 98]]),
    (chl_water,  "Three-band Chl-a", "YlGn",    *[np.nanpercentile(chl_water,  p) for p in [2, 98]]),
    (fai_water,  "FAI",              "BuPu",    *[np.nanpercentile(fai_water,  p) for p in [2, 98]]),
    (pc_water,   "Phycocyanin",      "Oranges", *[np.nanpercentile(pc_water,   p) for p in [2, 98]]),
    (flh_water,  "FLH",              "YlOrRd",  *[np.nanpercentile(flh_water,  p) for p in [2, 98]]),
    (cdom_water, "CDOM proxy",       "YlOrBr",  *[np.nanpercentile(cdom_water, p) for p in [2, 98]]),
]

fig, axes = plt.subplots(1, len(indices), figsize=(5 * len(indices), 8))

for ax, (data, title, cmap, vmin, vmax) in zip(axes, indices):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

plt.suptitle("Hyperion water quality indices — Lake Garda, 7 Oct 2002", fontsize=13)
plt.tight_layout()
plt.show()
